[reference](https://github.com/open-mmlab/mmaction2/blob/main/mmaction/models/backbones/resnet3d.py)

In [1]:
import sys

sys.path.append('../../../')

In [2]:
from __future__ import annotations

import copy
import warnings
from typing import Union, Optional, Sequence

import torch
import torch.nn as nn
import torch.utils.checkpoint as cp

%load_ext autoreload
%autoreload 2

from computer_vision.slowfast.mmcv.cnn.bricks.conv_module import ConvModule
from computer_vision.slowfast.mmcv.cnn.brick import build_activation_layer
from computer_vision.slowfast.mmaction.models.backbones.resnet3d import BasicBlock3d, Bottleneck3d
from computer_vision.slowfast.mmaction.models.backbones.resnet3d_slowfast import DeConvModule

In [3]:
spatial_stride=2
temporal_stride=1
downsample=ConvModule(8, 16, kernel_size=1, stride=(temporal_stride, spatial_stride, spatial_stride),
                      bias=False, conv_cfg=dict(type='Conv3d'), norm_cfg=dict(type='BN3d'),act_cfg=None)

block=BasicBlock3d(inplanes=8, planes=16, spatial_stride=spatial_stride, temporal_stride=temporal_stride, dilation=1,
                downsample=downsample, style='pytorch', inflate=True, non_local=False,
                non_local_cfg=dict(), conv_cfg=dict(type='Conv3d'), norm_cfg=dict(type='BN3d'), 
                 act_cfg=dict(type='ReLU'), with_cp=False, inflate_style='3x1x1')
print(block)
x=torch.rand(3,8,10,60,60)
out=block(x)
nn.MSELoss()(out, torch.rand_like(out)).backward()

BasicBlock3d(
  (conv1): ConvModule(
    (conv): Conv3d(8, 16, kernel_size=(3, 3, 3), stride=(1, 2, 2), padding=(1, 1, 1), bias=False)
    (bn): BatchNorm3d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (activation): ReLU(inplace=True)
  )
  (conv2): ConvModule(
    (conv): Conv3d(16, 16, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1), bias=False)
    (bn): BatchNorm3d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  )
  (downsample): ConvModule(
    (conv): Conv3d(8, 16, kernel_size=(1, 1, 1), stride=(1, 2, 2), bias=False)
    (bn): BatchNorm3d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  )
  (relu): ReLU()
)


In [4]:
spatial_stride=2
temporal_stride=1
downsample=ConvModule(32, 16*Bottleneck3d.expansion, kernel_size=1, stride=(temporal_stride, spatial_stride, spatial_stride),
                      bias=False, conv_cfg=dict(type='Conv3d'), norm_cfg=dict(type='BN3d'),act_cfg=None)
block=Bottleneck3d(32,16, spatial_stride=spatial_stride, temporal_stride=temporal_stride, dilation=1,
                downsample=downsample, style='pytorch', inflate=True, inflate_style='3x1x1',
                non_local=False, non_local_cfg=dict(), conv_cfg=dict(type='Conv3d'), 
                 norm_cfg=dict(type='BN3d'), act_cfg=dict(type='ReLU'), with_cp=False)
print(block)
x=torch.rand(3,32,10,60,60)
print(f"{x.shape=}, ({x.min().item()=},{x.max().item()=})")
out=block(x)
nn.MSELoss()(out, torch.rand_like(out)).backward()

Bottleneck3d(
  (conv1): ConvModule(
    (conv): Conv3d(32, 16, kernel_size=(3, 1, 1), stride=(1, 1, 1), padding=(1, 0, 0), bias=False)
    (bn): BatchNorm3d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (activation): ReLU(inplace=True)
  )
  (conv2): ConvModule(
    (conv): Conv3d(16, 16, kernel_size=(1, 3, 3), stride=(1, 2, 2), padding=(0, 1, 1), bias=False)
    (bn): BatchNorm3d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (activation): ReLU(inplace=True)
  )
  (conv3): ConvModule(
    (conv): Conv3d(16, 64, kernel_size=(1, 1, 1), stride=(1, 1, 1), bias=False)
    (bn): BatchNorm3d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  )
  (downsample): ConvModule(
    (conv): Conv3d(32, 64, kernel_size=(1, 1, 1), stride=(1, 2, 2), bias=False)
    (bn): BatchNorm3d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  )
  (relu): ReLU()
)
x.shape=torch.Size([3, 32, 10, 60, 60]), (x.min().item()=2.384

In [5]:
module=DeConvModule(32,16,1,1)
print(f"{module=}")
x=torch.rand(3,32,10,24,24)
out=module(x)
nn.MSELoss()(out, torch.rand_like(out)).backward()

module=DeConvModule(
  (conv): ConvTranspose3d(32, 16, kernel_size=(1, 1, 1), stride=(1, 1, 1), bias=False)
  (bn): BatchNorm3d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU()
)
